# Day 33: Weekly Skill Checkpoint: RAG Pipelines

Two small upgrades first: hide **private data** and **re-rank** the chunks. Then we **finish** the project, let **other people** test it, and **fix** what they find.

## 🎯 The goal of today

**"I can build a working RAG system that answers questions from real documents."**

```
TOP     Part 6   Checkpoint                 show it + explain it
          ▲
        Part 5   Polish                     fix what your partner found
          ▲
        Part 4   Peer test                  a partner tests your bot
          ▲
        Part 3   Re-ranking                 retrieve 6, keep the best 3
          ▲
        Part 2   Hide private data (PII)    mask the CNIC and phone numbers
          ▲
BOTTOM  Part 1   The final app              RAG + Streamlit together
```

### New words today

| Word | Meaning |
|---|---|
| **Peer** | A classmate. |
| **Peer test** | A classmate uses your bot and tries to find problems. |
| **Bug** | A mistake in the program: a wrong answer, an error, a crash. |
| **Polish** | Small fixes that make the project better and nicer to use. |
| **PII** | Private facts about one person: CNIC, phone, email, card number. |
| **Mask** | Replace a private value with a label, like `[CNIC]`. |
| **Re-rank** | Put the retrieved chunks in a better order with a second, more careful score. |

---
# Part 1: The final app

We join **Day 27** (the Streamlit chat page) and **Day 32** (RAG version 2) in one file.

- The top of the file is the **Day 32 code**: load, chunk, store, retrieve, `rag`.
- The bottom is the **Day 27 page**, but the reply now comes from `rag(question)`.
- The `chai_db` folder keeps the vectors. The first run embeds the chunks once.

Run the cell. Then in the **terminal**, in this folder: `streamlit run chai_wala_rag_app.py`

In [ ]:
%%writefile chai_wala_rag_app.py
import os
from pathlib import Path
import streamlit as st
import chromadb
from dotenv import load_dotenv
from openai import OpenAI

MODEL = "nvidia/nemotron-3.5-lightning:free"
load_dotenv()
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPEN_ROUTER_KEY"))

def ask(messages, temperature=0):
    try:
        r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature)
        return (r.choices[0].message.content or "").strip()
    except Exception as e:                    # no key? rate limit (429)? wrong model name?
        return f"ERROR: {e}"

EMB_MODEL = "nvidia/nemotron-3-embed-1b:free"      # embedding model (free, Day 24)

def embed(texts):
    r = client.embeddings.create(model=EMB_MODEL, input=texts)    # one request for the whole list
    vectors = []
    for item in r.data:
        vectors.append(item.embedding)                            # one vector per text
    return vectors

from pathlib import Path

def load_documents(folder):
    docs = []
    for path in sorted(Path(folder).glob("*.txt")):              # every .txt file in the folder
        docs.append({
            "source": path.name,
            "text": path.read_text(encoding="utf-8")
        })
    return docs

def chunk_by_paragraph(text):
    paragraphs = []
    for p in text.split("\n\n"):                                  # a blank line = a new paragraph
        p = p.strip()
        if len(p) > 30:                                           # skip very short parts, like titles
            paragraphs.append(p)
    return paragraphs

def chunk_documents(docs):
    chunks = []
    for doc in docs:
        for i, text in enumerate(chunk_by_paragraph(doc["text"])):
            chunks.append({
                "id": f"{doc['source']}-{i}",
                "source": doc["source"],
                "text": text
            })
    return chunks

chunks = chunk_documents(load_documents("chai_wala_docs"))

import chromadb

db = chromadb.PersistentClient(path="chai_db")              # saved in the folder "chai_db": not lost
shop = db.get_or_create_collection(
    name="chai_wala_docs",
    metadata={
        "hnsw:space": "cosine"
    }
)

if shop.count() == 0:                                       # embed only the first time (saves free requests)
    ids = []
    documents = []
    metadatas = []
    for c in chunks:
        ids.append(c["id"])
        documents.append(c["text"])
        metadatas.append({      # keep the file name with each chunk
            "source": c["source"]
        })

    shop.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
        embeddings=embed(documents),
    )

def retrieve(question, n=3):
    r = shop.query(query_embeddings=embed([question]), n_results=n)
    texts = r["documents"][0]          # [0] = results for our first (only) question
    sources = r["metadatas"][0]
    distances = r["distances"][0]

    found = []
    for i in range(len(texts)):
        found.append({
            "text": texts[i],
            "source": sources[i]["source"],
            "distance": distances[i]
        })
    return found

I_DONT_KNOW = "Sorry, I don't know. Please call 042-1234567."
CUTOFF = 0.85                    # choose it by looking at YOUR distances (Step 4)

RAG_SYSTEM = f"""You are the support bot of Chai Wala, a tea shop in Gulberg, Lahore.
Rules:
- Answer using ONLY the context. Never make up a price, place or time.
- If the answer is not in the context, reply exactly: "{I_DONT_KNOW}"
- At the end, write the source file in brackets, like [menu.txt].
- Be friendly. Reply in at most 3 short sentences."""

def rag(question):
    found = retrieve(question)                                          # 1. Retrieve
    if found[0]["distance"] > CUTOFF:                                   # nothing is close: do not ask the LLM
        return I_DONT_KNOW
    pieces = []
    for f in found:
        pieces.append(f"[{f['source']}] {f['text']}")               # each chunk with its source
    context = "\n\n".join(pieces)
    messages = [        # 2. Augment
        {
            "role": "system",
            "content": RAG_SYSTEM
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }
    ]
    return ask(messages)                                                # 3. Generate

# ---------------- the web page (Day 27) ----------------
st.title("☕ Chai Wala Support Bot")
st.caption("Answers come from the Chai Wala documents. The source file is shown in [brackets].")

if "messages" not in st.session_state:
    st.session_state.messages = [{
        "role": "assistant",
        "content": "Assalam o Alaikum! Ask me about our menu, delivery, timings or refunds."
    }]

for m in st.session_state.messages:
    st.chat_message(m["role"]).write(m["content"])

question = st.chat_input("Ask Chai Wala...")
if question:
    st.chat_message("user").write(question)
    st.session_state.messages.append({
        "role": "user",
        "content": question
    })

    reply = rag(question)                                    # RAG version 2 (Day 32)

    st.chat_message("assistant").write(reply)
    st.session_state.messages.append({
        "role": "assistant",
        "content": reply
    })

**One limit to know:** `rag` only looks at the **last** question. So a follow-up like *"And for 2 cups?"* does not work well. Retrieve does not know what "2 cups" of **what**. Real systems fix this by first rewriting the question with the chat history. Ask your customers to write full questions for now.

**✏️ Quick Check:** In the app, change the welcome message so it also says the phone number 042-1234567. Where do you change it?

In [ ]:
# your code here

**Answer:** In the line `st.session_state.messages = [{"role": "assistant", "content": "Assalam o Alaikum! ..."}]`. Add the number to the text, save the file, and click **Rerun**. (Then click the menu → **Clear cache** or open a new tab, because the old chat is still in `session_state`.)

---
# Part 2: Hide private data (PII)

**PII** = **P**ersonal **I**dentifiable **I**nformation. Facts that point to **one person**: CNIC number, phone number, email, card number, home address.

**Why it matters:** the question goes to the **LLM company**, and into our **logs**. An employee may type *"My CNIC is 35202-1234567-1 ... is my leave approved?"*. We should not send the CNIC anywhere. The LLM does **not** need it to answer.

*Everyday example:* a photocopy shop. Before it sends your form to the office, it covers your CNIC number with a **black marker**. The office still reads the form, but your number is safe.

### 4 ways to handle PII

| Technique | What we do | Example | When |
|---|---|---|---|
| **Block** | Reject the whole message | *"Please do not type your CNIC."* | Day 32 guardrail. Simple, but the user must type again. |
| **Mask** (redact) | Replace the private part with a **label** | `35202-1234567-1` → `[CNIC]` | The answer does not need the value. **Most common.** |
| **Swap and restore** (pseudonymize) | Replace with a **fake tag**, keep the real value in a small table, put it **back** in the answer | `0300-1234567` → `<PHONE_1>` → back to `0300-1234567` | The answer must show the value again. |
| **Do not keep** | Never save the private part in logs or the database | Save `[CNIC]`, not the number | Always, for logs. |

**Where does it go in the pipeline?** At the **very start**, in the input guardrail. **Before** the LLM and **before** the logs.

In [ ]:
import re

PII_PATTERNS = {                       # the private things we look for
    "CNIC":  r"(?<!\d)\d{5}-?\d{7}-?\d(?!\d)",            # 35202-1234567-1
    "PHONE": r"(?<!\d)(?:\+92|0)3\d{2}-?\d{7}(?!\d)",      # 0300-1234567 or +923001234567
    "EMAIL": r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+",               # ali.khan@gmail.com
    "CARD":  r"(?<!\d)(?:\d[ -]?){13,16}(?!\d)",            # 4111 1111 1111 1111
}

def mask_pii(text):                    # technique 2: MASK
    for name, pattern in PII_PATTERNS.items():
        text = re.sub(pattern, f"[{name}]", text)
    return text

message = "My CNIC is 35202-1234567-1 and my phone is 0300-1234567. Is my leave of 3 days approved?"
print(mask_pii(message))

The LLM now sees *"My CNIC is [CNIC] and my phone is [PHONE]. Is my leave of 3 days approved?"*. It can still answer. The private numbers never left our computer.

Now **swap and restore**. The answer should say *"We will call you on 0300-1234567"*, but the LLM must **not** see the real number:

In [ ]:
def hide_pii(text):                    # technique 3: SWAP. Keep the real values in a small table
    secret = {}                        # fake tag -> real value
    for name, pattern in PII_PATTERNS.items():
        for i, value in enumerate(re.findall(pattern, text), 1):
            tag = f"{name}_{i}"
            secret[tag] = value
            text = text.replace(value, f"<{tag}>")
    return text, secret

def show_pii(text, secret):            # ... and RESTORE them in the answer
    for tag, value in secret.items():
        text = text.replace(f"<{tag}>", value)
    return text

safe_text, secret = hide_pii("Please call me on 0300-1234567 about my leave.")
print(safe_text)                       # this goes to the LLM
print(secret)                          # this stays on our computer

llm_answer = "Thanks! We will call you on <PHONE_1> about your leave."      # what the LLM writes
print(show_pii(llm_answer, secret))                                          # what the user sees

**Be honest about the limits:** our patterns find only the formats we wrote. A **name** (*"Ali Khan"*) or an **address** has no fixed pattern. Real systems add a **model** that finds names and addresses (for example the free tool *Presidio*). Rules + a model, like the guardrail on Day 32.

**✏️ Quick Check:** Why is **masking** better than **blocking** for *"My CNIC is 35202-1234567-1. Is my leave approved?"*

In [ ]:
# your code here

**Answer:** Blocking throws the whole question away, and the employee must type it again. Masking removes only the **private part** (`[CNIC]`), and the bot still answers the real question.

---
# Part 3: Re-ranking

**The problem:** retrieve finds the 3 closest chunks. But "closest in meaning" is only a **rough** score. The best chunk may be **4th or 5th**, and then the bot never sees it.

**The fix:** two steps.

1. **Retrieve more:** ask Chroma for **6** chunks (`n=6`). Fast, but rough.
2. **Re-rank:** a **second, more careful** score puts the best chunk first. Keep only the **top 3** for the LLM.

*Everyday example:* hiring. HR **shortlists 6 CVs** quickly. Then the manager **reads each one carefully** and picks the best 3.

```
question → retrieve 6 → RE-RANK → best 3 → LLM → answer
```

Below are 6 chunks, in the order retrieve gave them for *"Do I get extra money if I work after 5 pm?"*. The right chunk (overtime) is **5th**. With `n=3` it would be **lost**.

In [ ]:
question = "Do I get extra money if I work after 5 pm?"

found = [        # as if retrieve(question, n=6) returned them (best first, in the rough order)
    {"source": "working_hours.txt", "text": "The office is open from 9 am to 5 pm, Monday to Friday. The office is closed on Saturday and Sunday."},
    {"source": "expense_policy.txt", "text": "The hotel limit is Rs 15000 per night. The food limit is Rs 2000 per day."},
    {"source": "working_hours.txt", "text": "The lunch break is 1 hour, from 1 pm to 2 pm. Please do not book meetings in this time."},
    {"source": "working_hours.txt", "text": "If you arrive after 9:30 am, you are marked late. Three late marks in one month cut half a day of salary."},
    {"source": "working_hours.txt", "text": "Work after 5 pm is overtime. Overtime needs approval from your manager and is paid at 1.5 times the normal rate."},
    {"source": "remote_work.txt", "text": "The company pays Rs 3000 each month for your home internet. Send a photo of your internet bill to claim it."},
]

print("Before (rough order):")
for i, f in enumerate(found[:3]):               # what the LLM would get with n=3
    print(f"  {i + 1}. [{f['source']}] {f['text'][:50]}")

The LLM would **not** see the overtime rule. Now the re-ranker. Ours is **simple**: count how many **important question words** are in each chunk (we skip small words like *do*, *the*, *if*). More shared words = higher score. Chunks with the same score keep their old order.

In [ ]:
STOP = {"do", "i", "get", "if", "the", "a", "an", "is", "are", "my", "me", "to", "of", "for", "in", "on",
        "and", "or", "can", "how", "what", "does", "it", "you", "your"}

def words(text):
    return {w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP}

def rerank(question, found, top=3):
    q = words(question)
    scored = sorted(found, key=lambda f: len(q & words(f["text"])), reverse=True)   # most shared words first
    return scored[:top]

print("After re-ranking:")
for i, f in enumerate(rerank(question, found)):
    print(f"  {i + 1}. [{f['source']}] {f['text'][:50]}")

The overtime chunk is now **first**. In `rag`, the change is small: `found = rerank(question, retrieve(question, n=6))`.

**Our scorer is a toy.** It only matches **exact words**: *"money"* does not match *"paid"*. Real systems use a **model that reads the question and the chunk together**:

| Re-ranker | How | Cost |
|---|---|---|
| **Words** (what we built) | Count shared words | Free, instant |
| **Cross-encoder** (for example `bge-reranker`) | A small model scores each (question, chunk) pair | Free to run on your computer, a little slow |
| **Rerank API** (for example Cohere Rerank) | The same idea, as a paid service | Paid, very good |
| **LLM** | Ask the LLM: *"Rank these chunks"* | 1 extra request per question |

In the Day 32 picture, this is the **"Improve ranking"** box.

**✏️ Quick Check:** Why do we retrieve **6** chunks and keep **3**, and not just retrieve 3?

In [ ]:
# your code here

**Answer:** The best chunk may be 4th, 5th or 6th in the rough order. If we retrieve only 3, it is lost. With 6 we **catch** it, the re-ranker puts it first, and we still send **only 3** to the LLM (short prompt, fewer wrong chunks).

---
# Part 4: Peer test

**How it works:**

1. Sit with a partner. Open **your** app on **your** laptop.
2. Your partner asks **8 questions**, one of each kind in the table below. **You do not help.**
3. You write down every answer. Mark it ✅ (right) or ❌ (wrong).
4. Then switch.

**The 8 kinds of questions:**

| # | Kind | Example for Chai Wala |
|---|---|---|
| 1 | An easy fact | *How much is a samosa?* |
| 2 | Different words | *What does it cost to bring chai to my house?* |
| 3 | A "no" in the documents | *Do you have chicken samosa?* |
| 4 | Two facts together | *Are you open at 1:30 pm on Friday?* |
| 5 | Numbers | *10 cups of Kashmiri chai: is delivery free?* |
| 6 | A spelling mistake | *dilivery charges?* |
| 7 | Roman Urdu | *Refund kaise milega?* |
| 8 | Not in the documents / not about the shop | *Is there parking?* / *Who won the match?* |

**Your test sheet** (copy it into a markdown cell, and fill it in):

| # | Question | Bot answer (short) | Source shown? | ✅ / ❌ |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |
| 6 | | | | |
| 7 | | | | |
| 8 | | | | |

**Score:** ___ / 8

**Remember Day 25:** the old bot broke on kinds 2, 6 and 8. Does yours?

---
# Part 5: Polish

Take your ❌ answers. For each one, use `retrieve(question)` to see what was found (Day 31). Then find it in this table:

| What you see | Why | Fix | Day |
|---|---|---|---|
| The right chunk is **not** found | Bad chunks, or the question uses very different words | Cut the documents better. Try `n=5`. | 30, 31 |
| The right chunk is found, but the answer is wrong | The prompt is not clear | Add a rule to `RAG_SYSTEM` | 26, 32 |
| It makes up a fact | The rule "ONLY the context" is weak or missing | Make the rule strict. Check the cutoff. | 32 |
| It says "I don't know", but the answer **is** in the documents | `CUTOFF` is too small | Print the distances. Raise the cutoff a little. | 32 |
| Wrong maths | LLMs are weak at maths | Add *"Show the calculation step by step."* | 32 |
| Answers in English to Roman Urdu | No language rule | Add *"Reply in the same language as the customer."* | 26 |
| No `[source]` at the end | The rule is missing or ignored | Check the rule. Put the sources in the context. | 32 |
| Error 429 | The free daily limit | Wait. Do not run the test again and again. | 24 |
| A new document is not used | The database was built before it was added | Add its chunks with `shop.add`, or delete `chai_db` and run again | 31 |

**Rule:** after **every** fix, ask the **same 8 questions** again. A fix for one question can break another one.

In [ ]:
# Part 5: your fixes here (copy the code you change, and test it)

---
# Part 6: ✅ Weekly Skill Checkpoint

**"I can build a working RAG system that answers questions from real documents."**

### Build (show your teacher)

- [ ] `streamlit run ...` opens your RAG app.
- [ ] It loads **real documents** from a folder, and cuts them into chunks.
- [ ] The chunks are stored in **Chroma** on disk (`PersistentClient`).
- [ ] It answers a question from the documents, and shows the **source**.
- [ ] It says **"I don't know"** for a question that is not in the documents.
- [ ] Your peer test score is **6 / 8 or better** (after polish).
- [ ] The key is in `.env`, not in the code.
- [ ] Private numbers (CNIC, phone) are **masked** before the question goes to the LLM.

### Explain (write 1–2 lines for each)

1. What do the 3 letters of **RAG** mean? Say each step for the Chai Wala bot.
2. Why do we **chunk** the documents? What goes wrong if a chunk is too big? Too small?
3. What does Chroma store for each chunk? Why do we save the **source**?
4. Why do we use `PersistentClient` and `if shop.count() == 0`?
5. Your bot gives a wrong answer. What do you look at **first**, and why?
6. What is **PII**? Name 2 ways to protect it.
7. What does **re-ranking** do? Why do we retrieve more chunks first?
8. Your manager says: *"Let's fine-tune a model on our menu."* What do you answer?

In [ ]:
# Explain: write your answers here (or in a markdown cell)

### Key takeaways

- A project is not done when it works for **you**. It is done when it works for **other people**.
- Test with **different kinds** of questions, not only easy ones.
- When an answer is wrong, find **which part** broke: retrieve, or generate.
- After every fix, run **all** the tests again.